[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module4/Multivariate_Occupancy_RNN_AdvancedTopics.ipynb)

# Multivariate Example (Advanced Topics)
**Dr. Dave Wanik - University of Connecticut**

Let's see if we can predict whether a room is occupied as a function of its environmental data - let's see if Conv1D and MaxPooling1D can make an even better prediction (ConvLSTM).

**Common errors:** forgetting an activation function, not specifying the input shape (or mixing up the order!), not returning sequences when two layers (or accidentally return_sequences=True when only one layer!)


In [1]:
# standard modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# RNN-specific modules
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report,accuracy_score
from tensorflow.keras import layers, Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D
from tensorflow.keras.layers import Dense, Dropout, SimpleRNN, GRU, LSTM, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

# reproducibility: same seed every run (numbers on CPU match exactly; a GPU may drift a little)
import keras
keras.utils.set_random_seed(5509)


## Read in data
Check for missing values, make some plots.

In [2]:
# Dataset: UCI room-occupancy detection (Candanedo & Feldheim, 2016), originally from LuisM78's GitHub.
# We use the two real files - 8,143 minutes of training data (Feb 4-10) followed by 9,752 of test (Feb 11-18) -
# stacked in date order into one two-week series. (The 2-day datatest.txt slice was too short to learn from.)
base = "https://raw.githubusercontent.com/drdave-teaching/OPIM5509Files/main/OPIM5509_Module4_Files/data/"
df = pd.concat([pd.read_csv(base + "datatraining.txt"), pd.read_csv(base + "datatest2.txt")], ignore_index=True)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)   # chronological - never shuffle a time series
print(df.info())
df.head(n=15) # nice complete data! this will allow us to check our work later

<class 'pandas.DataFrame'>
RangeIndex: 17895 entries, 0 to 17894
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           17895 non-null  datetime64[us]
 1   Temperature    17895 non-null  float64       
 2   Humidity       17895 non-null  float64       
 3   Light          17895 non-null  float64       
 4   CO2            17895 non-null  float64       
 5   HumidityRatio  17895 non-null  float64       
 6   Occupancy      17895 non-null  int64         
dtypes: datetime64[us](1), float64(5), int64(1)
memory usage: 978.8 KB
None


,date,Temperature,Humidity,Light,CO2,HumidityRatio,Occupancy
0,2015-02-04 17:51:00,23.180,27.272000,426.0,721.250000,0.004793,1
1,2015-02-04 17:51:59,23.150,27.267500,429.5,714.000000,0.004783,1
2,2015-02-04 17:53:00,23.150,27.245000,426.0,713.500000,0.004779,1
3,2015-02-04 17:54:00,23.150,27.200000,426.0,708.250000,0.004772,1
4,2015-02-04 17:55:00,23.100,27.200000,426.0,704.500000,0.004757,1
5,2015-02-04 17:55:59,23.100,27.200000,419.0,701.000000,0.004757,1
6,2015-02-04 17:57:00,23.100,27.200000,419.0,701.666667,0.004757,1
7,2015-02-04 17:57:59,23.100,27.200000,419.0,699.000000,0.004757,1
8,2015-02-04 17:58:59,23.100,27.200000,419.0,689.333333,0.004757,1
9,2015-02-04 18:00:00,23.075,27.175000,419.0,688.000000,0.004745,1


In [3]:
# count of occupancy
df['Occupancy'].value_counts() # not perfectly balanced, but that's OK

Occupancy
0    14117
1     3778
Name: count, dtype: int64

In [4]:
# visualize the data
df['Occupancy'].plot()
plt.show()

C:\Users\dww05002\AppData\Local\Temp\ipykernel_26120\633319714.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# visualize the data
df['CO2'].plot()
plt.show()

C:\Users\dww05002\AppData\Local\Temp\ipykernel_26120\165701153.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# drop the date column
df.drop(['date'], inplace=True, axis=1)
print(df.shape)
df.head()

(17895, 6)


,Temperature,Humidity,Light,CO2,HumidityRatio,Occupancy
0,23.18,27.2720,426.0,721.25,0.004793,1
1,23.15,27.2675,429.5,714.00,0.004783,1
2,23.15,27.2450,426.0,713.50,0.004779,1
3,23.15,27.2000,426.0,708.25,0.004772,1
4,23.10,27.2000,426.0,704.50,0.004757,1


In [7]:
# prep data for modeling (multivariate)
# link: https://machinelearningmastery.com/how-to-develop-lstm-models-for-time-series-forecasting/

from numpy import array

# split a multivariate sequence into samples
def split_sequences(sequences, n_steps):
	X, y = list(), list()
	for i in np.arange(len(sequences)): # be careful of this line!
		# find the end of this pattern
		end_ix = i + n_steps
		# check if we are beyond the dataset
		if end_ix > len(sequences):
			break
		# gather input and output parts of the pattern
		seq_x, seq_y = sequences[i:end_ix, :-1], sequences[end_ix-1, -1]
		X.append(seq_x)
		y.append(seq_y)
	return np.array(X), np.array(y)

In [8]:
# we could split our data first, normalize it, then create sequences

In [9]:
# all we need to do is decide on is n_steps (what our lookback period is)
# since we have a bunch of data, why not n_steps=10? then try 30 later on.
n_steps = 50
raw_seq = np.array(df) #make sure your data is stored as a numpy array!
# let's ignore the date column and just use the temperature data
X, y = split_sequences(raw_seq, n_steps=50)

In [10]:
# take a peak at what it did
print(X.shape)
print(y.shape)

# scroll up and make sure you understand this!
# y is a function of X (the previous n_steps observations!)

(17846, 50, 5)
(17846,)


In [11]:
# split the data into train and test partitions
# we will use 50% of the data for train, and 50% for validation
train_pct_index = int(0.5 * len(X))
X_train, X_test = X[:train_pct_index], X[train_pct_index:]
y_train, y_test = y[:train_pct_index], y[train_pct_index:]

# pretty slick way of splitting your data using slicing!
# notice how we didn't do any shuffling (we don't want temporal leakage! keeps time series intact)

In [12]:
# check the shape to be sure
print(X.shape, X_train.shape, X_test.shape)
print(y.shape, y_train.shape, y_test.shape)

# verify that this all adds up!
# 2635 samples with 30 lookback and 6 columns

(17846, 50, 5) (8923, 50, 5) (8923, 50, 5)
(17846,) (8923,) (8923,)


# RNN one layer model (with Conv1D and pooling!)
With Conv1D and MaxPooling1D... don't forget that input shape needs to go in the first layer!

In [13]:
n_steps = X_train.shape[1] # lookback
n_features = X_train.shape[2] # columns

print(n_steps, n_features)

50 5


In [14]:
# now let's build a model

# define
n_steps = X_train.shape[1]
n_features = X_train.shape[2]


# define model
model = Sequential()
model.add(Conv1D(filters=32, kernel_size=3, input_shape=(n_steps,n_features))) # notice how input shape goes in first layer
model.add(MaxPooling1D(2))
model.add(SimpleRNN(30, activation='relu', recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['acc'])
model.summary()

es = EarlyStopping(monitor='val_acc', mode='max',
                   patience=10,
                   verbose=1,
                   restore_best_weights=True)

# fit model (uses early stopping)
model.fit(X_train, y_train,
          epochs=500,
          batch_size=64,
          validation_split=0.2, # val is a random 20% of the data since we set shuffle = True
          verbose=1,
          callbacks=[es],
          shuffle=True)

C:\Users\dww05002\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 48, 32)         │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 30)             │         1,890 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,433 (9.50 KB)

 Trainable params: 2,433 (9.50 KB)

...

 98/112 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - acc: 0.9766 - loss: 0.1046

106/112 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - acc: 0.9760 - loss: 0.1080

112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - acc: 0.9752 - loss: 0.1081 - val_acc: 0.9938 - val_loss: 0.0480


Epoch 19: early stopping


Restoring model weights from the end of the best epoch: 9.


In [15]:
# show scatterplots of actual vs. predicted for train and test
# make a prediction
pred = model.predict(X_test)# the pred
print(pred) # round them!

pred = np.round(pred,0)
pred # run all if you get an error...

  1/279 ━━━━━━━━━━━━━━━━━━━━ 1:20 289ms/step

 21/279 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step    

 46/279 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

 72/279 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

 99/279 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

...

270/279 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

279/279 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

279/279 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


[[1.9347812e-11]
 [2.1251240e-11]
 [3.5617377e-11]
 ...
 [1.0000000e+00]
 [1.0000000e+00]
 [1.0000000e+00]]

array([[0.],
       [0.],
       [0.],
       ...,
       [1.],
       [1.],
       [1.]], shape=(8923, 1), dtype=float32)

In [16]:
# confusion matrix
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(confusion_matrix(y_test, pred)) # looks pretty good!
print(classification_report(y_test, pred))

[[6879  209]
 [  21 1814]]


              precision    recall  f1-score   support

         0.0       1.00      0.97      0.98      7088
         1.0       0.90      0.99      0.94      1835

    accuracy                           0.97      8923
   macro avg       0.95      0.98      0.96      8923
weighted avg       0.98      0.97      0.97      8923



In [17]:
# show timeseries plot on the train and validation data
plt.plot(np.arange(X_test.shape[0]), y_test, color='blue') # actual data
plt.plot(np.arange(X_test.shape[0]), pred, color='red') # predicted data
plt.suptitle('Test Results')
plt.xlabel('Time')
plt.ylabel('Occupied')
plt.show()

C:\Users\dww05002\AppData\Local\Temp\ipykernel_26120\774347038.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# put it all together for other models

# make a prediction
pred = model.predict(X_test)# the pred
print(pred) # round them!

pred = np.round(pred,0)
print(pred) # run all if you get an error...

# confusion matrix - put this at the top!
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(confusion_matrix(y_test, pred)) # looks pretty good!
print(classification_report(y_test, pred))

# show timeseries plot on the train and validation data
plt.plot(np.arange(X_test.shape[0]), y_test, color='blue') # actual data
plt.plot(np.arange(X_test.shape[0]), pred, color='red') # predicted data
plt.suptitle('Test Results')
plt.xlabel('Time')
plt.ylabel('Occupied')
plt.show()

  1/279 ━━━━━━━━━━━━━━━━━━━━ 11s 42ms/step

 19/279 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step  

 38/279 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

 57/279 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

 76/279 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

...

279/279 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

279/279 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


[[1.9347812e-11]
 [2.1251240e-11]
 [3.5617377e-11]
 ...
 [1.0000000e+00]
 [1.0000000e+00]
 [1.0000000e+00]]
[[0.]
 [0.]
 [0.]
 ...
 [1.]
 [1.]
 [1.]]
[[6879  209]
 [  21 1814]]


              precision    recall  f1-score   support

         0.0       1.00      0.97      0.98      7088
         1.0       0.90      0.99      0.94      1835

    accuracy                           0.97      8923
   macro avg       0.95      0.98      0.96      8923
weighted avg       0.98      0.97      0.97      8923



C:\Users\dww05002\AppData\Local\Temp\ipykernel_26120\1192869489.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# RNN two layer model
Don't forget to set return_sequences=True!

In [19]:
# now let's build a model

# define
n_steps = X_train.shape[1]
n_features = X_train.shape[2]


# define model
model = Sequential()
model.add(Conv1D(filters=32, kernel_size=3, input_shape=(n_steps,n_features))) # notice how input shape goes in first layer
model.add(MaxPooling1D(2))
model.add(Bidirectional(SimpleRNN(30, return_sequences=True, recurrent_dropout=0.2))) # don't forget to return_sequences!
model.add(Bidirectional(SimpleRNN(30, recurrent_dropout=0.2)))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['acc'])
model.summary()

es = EarlyStopping(monitor='val_acc', mode='max',
                   patience=10,
                   verbose=1,
                   restore_best_weights=True)

# fit model (uses early stopping)
model.fit(X_train, y_train,
          epochs=500,
          batch_size=64,
          validation_split=0.2, # val is a random 20% of the data since we set shuffle = True
          verbose=1,
          callbacks=[es],
          shuffle=True)

C:\Users\dww05002\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_1 (Conv1D)               │ (None, 48, 32)         │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 24, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 24, 60)         │         3,780 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 60)             │         5,460 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            61 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,813 (38.33 KB)

 Trainable params: 9,813 (38.33 KB)

...

107/112 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - acc: 0.9541 - loss: 0.1282

110/112 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - acc: 0.9553 - loss: 0.1252

112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - acc: 0.9557 - loss: 0.1243 - val_acc: 0.9832 - val_loss: 0.0389


Epoch 18: early stopping


Restoring model weights from the end of the best epoch: 8.


In [20]:
# put it all together for other models

# make a prediction
pred = model.predict(X_test)# the pred
print(pred) # round them!

pred = np.round(pred,0)
print(pred) # run all if you get an error...

# confusion matrix - put this at the top!
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(confusion_matrix(y_test, pred)) # looks pretty good!
print(classification_report(y_test, pred))

# show timeseries plot on the train and validation data
plt.plot(np.arange(X_test.shape[0]), y_test, color='blue') # actual data
plt.plot(np.arange(X_test.shape[0]), pred, color='red') # predicted data
plt.suptitle('Test Results')
plt.xlabel('Time')
plt.ylabel('Occupied')
plt.show()

  1/279 ━━━━━━━━━━━━━━━━━━━━ 4:21 940ms/step

 10/279 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step    

 19/279 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step

 29/279 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step

 38/279 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step

...

279/279 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

279/279 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step


[[0.00091724]
 [0.00091724]
 [0.00091724]
 ...
 [0.22778545]
 [0.30025035]
 [0.3288676 ]]


[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [0.]
 [0.]]
[[6744  344]
 [  35 1800]]
              precision    recall  f1-score   support

         0.0       0.99      0.95      0.97      7088
         1.0       0.84      0.98      0.90      1835

    accuracy                           0.96      8923
   macro avg       0.92      0.97      0.94      8923
weighted avg       0.96      0.96      0.96      8923



C:\Users\dww05002\AppData\Local\Temp\ipykernel_26120\1192869489.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# LSTM one layer model
Literally, just grab the code above and change SimpleRNN to LSTM and boom! you have a more sophisticated model.

In [21]:
# define shape
n_steps = X_train.shape[1]
n_features = X_train.shape[2]


# define model
model = Sequential()
model.add(Conv1D(filters=32, kernel_size=3, input_shape=(n_steps,n_features))) # notice how input shape goes in first layer
model.add(MaxPooling1D())
model.add(LSTM(30, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['acc'])
model.summary()

es = EarlyStopping(monitor='val_acc', mode='max',
                   patience=10,
                   verbose=1,
                   restore_best_weights=True)

# fit model (uses early stopping)
model.fit(X_train, y_train,
          epochs=500,
          batch_size=64,
          validation_split=0.2, # val is a random 20% of the data since we set shuffle = True
          verbose=1,
          callbacks=[es],
          shuffle=True)

C:\Users\dww05002\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 48, 32)         │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 24, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 30)             │         7,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,103 (31.65 KB)

 Trainable params: 8,103 (31.65 KB)

...

103/112 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - acc: 0.9583 - loss: 0.3157

109/112 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - acc: 0.9576 - loss: 0.3245

112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - acc: 0.9573 - loss: 0.3271 - val_acc: 0.9501 - val_loss: 0.5468


Epoch 11: early stopping


Restoring model weights from the end of the best epoch: 1.


In [22]:
# put it all together for other models

# make a prediction
pred = model.predict(X_test)# the pred
print(pred) # round them!

pred = np.round(pred,0)
print(pred) # run all if you get an error...

# confusion matrix - put this at the top!
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(confusion_matrix(y_test, pred)) # looks pretty good!
print(classification_report(y_test, pred))

# show timeseries plot on the train and validation data
plt.plot(np.arange(X_test.shape[0]), y_test, color='blue') # actual data
plt.plot(np.arange(X_test.shape[0]), pred, color='red') # predicted data
plt.suptitle('Test Results')
plt.xlabel('Time')
plt.ylabel('Occupied')
plt.show()

  1/279 ━━━━━━━━━━━━━━━━━━━━ 59s 213ms/step

 16/279 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step   

 32/279 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

 45/279 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

 59/279 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

...

279/279 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step

279/279 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step


[[1.0889709e-20]
 [1.1012359e-20]
 [1.3084985e-20]
 ...
 [0.0000000e+00]
 [0.0000000e+00]
 [0.0000000e+00]]


[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [0.]
 [0.]]
[[7019   69]
 [ 279 1556]]
              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98      7088
         1.0       0.96      0.85      0.90      1835

    accuracy                           0.96      8923
   macro avg       0.96      0.92      0.94      8923
weighted avg       0.96      0.96      0.96      8923



C:\Users\dww05002\AppData\Local\Temp\ipykernel_26120\1192869489.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# LSTM two layer model

In [23]:
# now let's build a model

# since this is a univariate problem, n_features will be 1 (we also defined this before)

# define model
model = Sequential()
model.add(Conv1D(filters=128, kernel_size=3, input_shape=(n_steps,n_features))) # notice how input shape goes in first layer
model.add(MaxPooling1D(2))
model.add(Bidirectional(LSTM(30,
                            return_sequences=True, # remember, if stacking layers, you need to return sequences!
                            activation='relu',
                            recurrent_dropout=0.2)))
model.add(GRU(20, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['acc'])
model.summary()

es = EarlyStopping(monitor='val_acc', mode='max',
                   patience=10,
                   verbose=1,
                   restore_best_weights=True)

# fit model (uses early stopping)
model.fit(X_train, y_train,
          epochs=500,
          batch_size=64,
          validation_split=0.2, # val is a random 20% of the data since we set shuffle = True
          verbose=1,
          callbacks=[es],
          shuffle=True)

C:\Users\dww05002\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_3 (Conv1D)               │ (None, 48, 128)        │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 24, 60)         │        38,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 20)             │         4,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 45,149 (176.36 KB)

 Trainable params: 45,149 (176.36 KB)

...

109/112 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - acc: 0.7721 - loss: 0.3194

111/112 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - acc: 0.7713 - loss: 0.3203

112/112 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - acc: 0.7712 - loss: 0.3202 - val_acc: 0.8207 - val_loss: 0.2179


Epoch 20: early stopping


Restoring model weights from the end of the best epoch: 10.


In [24]:
# put it all together for other models

# make a prediction
pred = model.predict(X_test)# the pred
print(pred) # round them!

pred = np.round(pred,0)
print(pred) # run all if you get an error...

# confusion matrix - put this at the top!
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(confusion_matrix(y_test, pred)) # looks pretty good!
print(classification_report(y_test, pred))

# show timeseries plot on the train and validation data
plt.plot(np.arange(X_test.shape[0]), y_test, color='blue') # actual data
plt.plot(np.arange(X_test.shape[0]), pred, color='red') # predicted data
plt.suptitle('Test Results')
plt.xlabel('Time')
plt.ylabel('Occupied')
plt.show()

  1/279 ━━━━━━━━━━━━━━━━━━━━ 6:49 1s/step

  6/279 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step

 11/279 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step

 16/279 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step

 21/279 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step

...

279/279 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step

279/279 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step


[[0.11493158]
 [0.11346414]
 [0.11674272]
 ...
 [0.33259243]
 [0.33798724]
 [0.21990974]]


[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [0.]
 [0.]]
[[7082    6]
 [1830    5]]
              precision    recall  f1-score   support

         0.0       0.79      1.00      0.89      7088
         1.0       0.45      0.00      0.01      1835

    accuracy                           0.79      8923
   macro avg       0.62      0.50      0.45      8923
weighted avg       0.72      0.79      0.70      8923



C:\Users\dww05002\AppData\Local\Temp\ipykernel_26120\1192869489.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Save the model and use it again

Reproducibility means more than a seed: **save the fitted model** so you (or a teammate, or your future self) can reload it and predict without retraining. Keras 3 saves to a single `.keras` file. The reloaded model must give *identical* predictions - we check.

In [25]:
from keras.models import load_model

model.save('Multivariate_Occupancy_RNN_AdvancedTopics.keras')                 # one file: architecture + weights + optimizer state
reloaded = load_model('Multivariate_Occupancy_RNN_AdvancedTopics.keras')

# same inputs, same answers?
import numpy as np
same = np.allclose(model.predict(X_test[:5], verbose=0), reloaded.predict(X_test[:5], verbose=0))
print('reloaded model reproduces the predictions:', same)
reloaded.summary()

reloaded model reproduces the predictions: True


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_3 (Conv1D)               │ (None, 48, 128)        │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 24, 60)         │        38,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 20)             │         4,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 135,449 (529.10 KB)

 Trainable params: 45,149 (176.36 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 90,300 (352.74 KB)